# baselines on dataset v2
shireen - build night 6

task: run histogram + WL on the new dataset, synthetic first then real, and get accuracy + time.

notes:
- using the graphs file (`graphs/<track>.npz`) + `samples.csv` (sample_id, function_id, role) instead of parsing the .bench files or making pytorch graphs
- near-miss circuits stay in as distractors since the eval uses them
- scoring the same way as the dataset's `evaluate.py` (eval queries + hard galleries)
- baselines don't train so the only thing to pick is how many WL rounds (h). picking that on val and only running test at the end

## setup

In [1]:
# where the dataset is (unzip circuitree_v2.zip here or use drive)
ROOT = "/content/circuitree_v2"
# from google.colab import drive; drive.mount("/content/drive")
# !unzip -q /content/drive/MyDrive/circuitree_v2.zip -d /content
import os; print(sorted(os.listdir(ROOT)))

['README.md', '__pycache__', 'baselines.csv', 'circuitree_data.py', 'circuits', 'eval', 'evaluate.py', 'example_train.py', 'functions.csv', 'graphs', 'pairs', 'quickstart_colab.ipynb', 'requirements.txt', 'samples.csv', 'stats.json']


In [2]:
import csv, os, sys, time, json
import numpy as np
import scipy.sparse as sp
import pandas as pd

## load data
only loads the circuits in the split we want

In [3]:
def load_split(root, track, split):
    # loads graphs + labels for one track/split
    # node_type: 0 = input, 1 = AND, 2 = output. edge_inv = 1 if the wire has a NOT on it
    rows = [r for r in csv.DictReader(open(os.path.join(root, "samples.csv"), newline=""))
            if r["track"] == track and r["split"] == split]
    with np.load(os.path.join(root, "graphs", f"{track}.npz")) as z:
        a = {k: z[k] for k in z.files}
    pos = {s: i for i, s in enumerate(a["sample_ids"].tolist())}
    idx = np.array([pos[r["sample_id"]] for r in rows])

    n_lo, n_hi = a["node_ptr"][idx], a["node_ptr"][idx + 1]
    e_lo, e_hi = a["edge_ptr"][idx], a["edge_ptr"][idx + 1]
    n_cnt, e_cnt = n_hi - n_lo, e_hi - e_lo
    node_sel = np.concatenate([np.arange(l, h) for l, h in zip(n_lo, n_hi)])
    edge_sel = np.concatenate([np.arange(l, h) for l, h in zip(e_lo, e_hi)])
    new_start = np.concatenate([[0], np.cumsum(n_cnt)[:-1]])
    edge_graph = np.repeat(np.arange(len(rows)), e_cnt)
    return dict(
        sample_id=[r["sample_id"] for r in rows],
        function_id=[r["function_id"] for r in rows],
        role=[r["role"] for r in rows],
        node_type=a["node_type"][node_sel].astype(np.int64),
        pi_index=a["pi_index"][node_sel].astype(np.int64),
        node_graph=np.repeat(np.arange(len(rows)), n_cnt),
        edge_src=a["edge_src"][edge_sel].astype(np.int64) + new_start[edge_graph],
        edge_dst=a["edge_dst"][edge_sel].astype(np.int64) + new_start[edge_graph],
        edge_inv=a["edge_inv"][edge_sel].astype(np.int64),
        edge_graph=edge_graph,
    )

## baseline 1 - histogram
same as week 3, count gates + wires, make them proportions, compare w/ cosine

In [4]:
def histogram_vectors(d):
    # counts [ANDs, inputs, outputs, normal wires, inverted wires] -> proportions
    # (NOTs are inverted wires in an AIG so this is basically the gate histogram)
    n = len(d["sample_id"])
    cnt = np.zeros((n, 5))
    for t in range(3):
        cnt[:, [1, 0, 2][t]] = np.bincount(d["node_graph"], weights=(d["node_type"] == t), minlength=n)
    inv = np.bincount(d["edge_graph"], weights=d["edge_inv"], minlength=n)
    cnt[:, 3] = np.bincount(d["edge_graph"], minlength=n) - inv
    cnt[:, 4] = inv
    return cnt / cnt.sum(axis=1, keepdims=True)

## baseline 2 - WL
grakel won't import with the numpy colab has now so I just wrote WL with numpy. this also does all the circuits at once which matters bc the real set has ~9.6M nodes

checked it against a slow normal version of WL on 149 circuits and the scores matched (diff ~4e-16)

`use_positions=True` = extra version where WL also knows which input is x0, x1... (README says input order matters)

In [5]:
def _mix(x):
    # scrambles a number into a random-looking 64 bit number, used to make WL labels
    x = (x + np.uint64(0x9E3779B97F4A7C15))
    x = (x ^ (x >> np.uint64(30))) * np.uint64(0xBF58476D1CE4E5B9)
    x = (x ^ (x >> np.uint64(27))) * np.uint64(0x94D049BB133111EB)
    return x ^ (x >> np.uint64(31))


def wl_vectors(d, h=3, use_positions=False):
    # WL: start with node type, then each round a node's label = its label + its neighbours' labels
    # (neighbours tagged with direction + whether the wire is inverted)
    # count labels from every round -> vector. cosine of 2 vectors = normalized WL kernel
    # use_positions=True also tells it which input is x0, x1, ...
    with np.errstate(over="ignore"):
        lab = d["node_type"].copy()
        if use_positions:
            lab = np.where(lab == 0, 3 + d["pi_index"], lab)
        n_nodes, n = len(lab), len(d["sample_id"])
        src, dst, inv = d["edge_src"], d["edge_dst"], d["edge_inv"]
        blocks = []
        for it in range(h + 1):
            _, lab = np.unique(lab, return_inverse=True)            # renumber labels 0,1,2...
            blocks.append(sp.csr_matrix((np.ones(n_nodes), (d["node_graph"], lab)),
                                        shape=(n, lab.max() + 1)))
            if it == h:
                break
            L = lab.astype(np.uint64)
            # what each node hears from its inputs and its outputs
            m_in = _mix(L[src] * np.uint64(4) + inv.astype(np.uint64) * np.uint64(2))
            m_out = _mix(L[dst] * np.uint64(4) + inv.astype(np.uint64) * np.uint64(2) + np.uint64(1))
            acc = np.zeros(n_nodes, dtype=np.uint64)           # sum so order doesn't matter
            np.add.at(acc, dst, m_in)
            np.add.at(acc, src, m_out)
            # hash own label + neighbours separately (fixed a bug where 2 diff nodes got the same label)
            lab = _mix(_mix(L | np.uint64(1 << 62)) ^ _mix(acc))
    X = sp.hstack(blocks).tocsr()
    norms = np.sqrt(np.asarray(X.multiply(X).sum(axis=1)).ravel())
    return sp.diags(1 / np.maximum(norms, 1e-12)) @ X

## scoring
same as `evaluate.py` but works with sparse vectors. next cell checks it gives the same numbers as the official one

In [6]:
def evaluate(root, track, split, vecs, sample_ids, condition="hard", seed=0):
    # same rules/metrics as the dataset's evaluate.py (checked, gives the same numbers)
    # but works with sparse vectors since the WL ones are huge
    rd = lambda name: list(csv.DictReader(open(os.path.join(root, name), newline="", encoding="utf-8")))
    samples = {r["sample_id"]: r for r in rd("samples.csv") if r["track"] == track and r["split"] == split}
    queries = rd(os.path.join("eval", f"{track}_{split}_queries.csv"))
    if condition == "hard":
        queries = [q for q in queries if q["in_hard_eval"] == "1"]
        gallery = {}
        for g in rd(os.path.join("eval", f"{track}_{split}_hard_gallery.csv")):
            gallery.setdefault(g["query_id"], []).append(g["candidate_id"])
    ids = list(sample_ids)
    index = {s: i for i, s in enumerate(ids)}
    M = sp.csr_matrix(vecs, dtype=np.float32) if sp.issparse(vecs) else np.asarray(vecs, dtype=np.float32)
    norms = np.sqrt(np.asarray(M.multiply(M).sum(axis=1)).ravel()) if sp.issparse(M) else np.linalg.norm(M, axis=1)
    M = (sp.diags(1 / (norms + 1e-12)) @ M) if sp.issparse(M) else M / (norms[:, None] + 1e-12)
    fn_code = {f: i for i, f in enumerate(sorted({samples[s]["function_id"] for s in ids}))}
    pg_code = {g: i for i, g in enumerate(sorted({samples[s]["perm_group_id"] for s in ids} - {""}))}
    fn = np.array([fn_code[samples[s]["function_id"]] for s in ids])
    pg = np.array([pg_code.get(samples[s]["perm_group_id"], -1) for s in ids])
    q_rows = np.array([index[q["query_id"]] for q in queries], dtype=int)
    sims = M[q_rows] @ M.T
    sims = sims.toarray() if sp.issparse(sims) else np.asarray(sims)
    rng = np.random.default_rng(seed)
    res = []
    for qi, q in enumerate(queries):
        r = q_rows[qi]
        cand = (np.array([index[c] for c in gallery[q["query_id"]]], dtype=int) if condition == "hard"
                else np.delete(np.arange(len(ids)), r))
        same = fn[cand] == fn[r]
        keep = same | (pg[cand] == -1) | (pg[cand] != pg[r])
        s, lab = sims[qi, cand[keep]], same[keep]
        order = np.lexsort((rng.random(len(s)), -s))
        hits = lab[order]
        p = np.flatnonzero(hits)
        if len(p) == 0:
            continue
        res.append(dict(hit1=float(hits[0]), mrr=1 / (p[0] + 1), r5=hits[:5].sum() / len(p),
                        r10=hits[:10].sum() / len(p), ap=float((np.arange(1, len(p) + 1) / (p + 1)).mean())))
    out = {k: round(float(np.mean([x[k] for x in res])), 3) for k in ["hit1", "mrr", "r5", "r10", "ap"]}
    out["queries"] = len(res)
    return out


def timed(fn, *args, **kw):
    t = time.perf_counter()
    out = fn(*args, **kw)
    return out, time.perf_counter() - t

In [7]:
# check: my scoring vs the official evaluate.py, should match
sys.path.insert(0, ROOT); import evaluate as official
d = load_split(ROOT, "synthetic", "test"); H = histogram_vectors(d)
for cond in ("hard", "full"):
    o = official.evaluate(dict(zip(d["sample_id"], H)), "synthetic", "test", cond, ROOT)
    m = evaluate(ROOT, "synthetic", "test", H, d["sample_id"], cond)
    print(cond, "official:", {k: round(o[k], 3) for k in ["hit1", "mrr", "r5", "r10", "ap"]}, "| mine:", {k: m[k] for k in ["hit1", "mrr", "r5", "r10", "ap"]})

hard official: {'hit1': 0.423, 'mrr': 0.588, 'r5': 0.234, 'r10': 0.393, 'ap': 0.36} | mine: {'hit1': 0.423, 'mrr': 0.588, 'r5': 0.234, 'r10': 0.393, 'ap': 0.36}


full official: {'hit1': 0.228, 'mrr': 0.356, 'r5': 0.105, 'r10': 0.154, 'ap': 0.114} | mine: {'hit1': 0.228, 'mrr': 0.356, 'r5': 0.105, 'r10': 0.154, 'ap': 0.114}


## run a track
1. load val + test
2. try h = 1-6 on val, keep the best avg mAP of hard + full (more rounds helps hard a tiny bit but kills full, every circuit ends up with its own labels)
3. run test once + time it

In [8]:
def run_track(track, hs=(1, 2, 3, 4, 5, 6)):
    data, load_s = {}, {}
    for split in ("val", "test"):
        data[split], load_s[split] = timed(load_split, ROOT, track, split)
    sweep, chosen = [], {}
    for pos in (False, True):
        best = None
        for h in hs:
            X = wl_vectors(data["val"], h=h, use_positions=pos)
            mh = evaluate(ROOT, track, "val", X, data["val"]["sample_id"], "hard")
            mf = evaluate(ROOT, track, "val", X, data["val"]["sample_id"], "full")
            sweep.append(dict(track=track, positions=pos, h=h, hard_hit1=mh["hit1"], hard_mAP=mh["ap"], full_hit1=mf["hit1"], full_mAP=mf["ap"]))
            score = (mh["ap"] + mf["ap"]) / 2  # avg of hard + full
            if best is None or score > best[1]:
                best = (h, score)
        chosen[pos] = best[0]
    d = data["test"]; n = len(d["sample_id"])
    methods = {"histogram": lambda: histogram_vectors(d),
               f"WL (h={chosen[False]})": lambda: wl_vectors(d, h=chosen[False]),
               f"WL + input positions (h={chosen[True]})": lambda: wl_vectors(d, h=chosen[True], use_positions=True)}
    rows = []
    for name, f in methods.items():
        X, t_feat = timed(f)
        for cond in ("hard", "full"):
            m, t_score = timed(evaluate, ROOT, track, "test", X, d["sample_id"], cond)
            rows.append(dict(track=track, split="test", condition=cond, method=name, queries=m["queries"],
                             hit1=m["hit1"], mrr=m["mrr"], r5=m["r5"], r10=m["r10"], ap=m["ap"],
                             vectors_ms_per_circuit=round(1000 * t_feat / n, 3),
                             rank_ms_per_query=round(1000 * t_score / m["queries"], 3)))
    print(f"{track}: loaded val in {load_s['val']:.1f}s, test in {load_s['test']:.1f}s ({n} test circuits)")
    return pd.DataFrame(rows), pd.DataFrame(sweep)

## synthetic

In [9]:
syn, syn_sweep = run_track("synthetic")
display(syn_sweep)
syn

synthetic: loaded val in 0.3s, test in 0.2s (1040 test circuits)


,track,positions,h,hard_hit1,hard_mAP,full_hit1,full_mAP
0,synthetic,False,1,0.733,0.491,0.609,0.243
1,synthetic,False,2,0.757,0.517,0.658,0.268
2,synthetic,False,3,0.767,0.523,0.629,0.227
3,synthetic,False,4,0.782,0.529,0.545,0.183
4,synthetic,False,5,0.792,0.534,0.460,0.145
5,synthetic,False,6,0.772,0.535,0.396,0.119
6,synthetic,True,1,0.812,0.566,0.723,0.332
7,synthetic,True,2,0.812,0.574,0.723,0.302
8,synthetic,True,3,0.797,0.579,0.629,0.220
9,synthetic,True,4,0.802,0.587,0.505,0.167


,track,split,condition,method,queries,hit1,mrr,r5,r10,ap,vectors_ms_per_circuit,rank_ms_per_query
0,synthetic,test,hard,histogram,201,0.423,0.588,0.234,0.393,0.360,0.001,1.718
1,synthetic,test,full,histogram,202,0.228,0.356,0.105,0.154,0.114,0.001,1.559
2,synthetic,test,hard,WL (h=2),201,0.821,0.879,0.403,0.526,0.533,0.018,1.939
3,synthetic,test,full,WL (h=2),202,0.713,0.753,0.269,0.313,0.296,0.018,2.311
4,synthetic,test,hard,WL + input positions (h=1),201,0.856,0.905,0.445,0.572,0.573,0.008,1.887
5,synthetic,test,full,WL + input positions (h=1),202,0.767,0.812,0.319,0.362,0.352,0.008,1.615


## real

In [10]:
real, real_sweep = run_track("real")
display(real_sweep)
real

real: loaded val in 2.9s, test in 0.7s (8887 test circuits)


,track,positions,h,hard_hit1,hard_mAP,full_hit1,full_mAP
0,real,False,1,0.627,0.570,0.433,0.232
1,real,False,2,0.650,0.586,0.432,0.225
2,real,False,3,0.656,0.591,0.371,0.186
3,real,False,4,0.661,0.595,0.315,0.152
4,real,False,5,0.670,0.598,0.275,0.126
5,real,False,6,0.675,0.600,0.235,0.107
6,real,True,1,0.693,0.631,0.577,0.329
7,real,True,2,0.711,0.643,0.519,0.248
8,real,True,3,0.721,0.644,0.413,0.168
9,real,True,4,0.726,0.646,0.315,0.119


,track,split,condition,method,queries,hit1,mrr,r5,r10,ap,vectors_ms_per_circuit,rank_ms_per_query
0,real,test,hard,histogram,1598,0.605,0.743,0.387,0.697,0.604,0.004,0.293
1,real,test,full,histogram,1652,0.368,0.456,0.144,0.187,0.159,0.004,2.133
2,real,test,hard,WL (h=2),1598,0.766,0.861,0.470,0.771,0.690,0.059,0.626
3,real,test,full,WL (h=2),1652,0.490,0.586,0.208,0.282,0.245,0.059,2.475
4,real,test,hard,WL + input positions (h=1),1598,0.795,0.883,0.508,0.831,0.737,0.036,0.812
5,real,test,full,WL + input positions (h=1),1652,0.679,0.760,0.344,0.476,0.429,0.036,2.480


## is it faster? (vs CEC)
CEC is always right but only checks 2 circuits at a time. timing it on 200 random pairs, then estimating how long it'd take to search a whole test set with it.

need ABC for this: `!apt-get -qq install -y yosys` (each call also includes starting ABC up)

In [11]:
import csv, random, subprocess, statistics
def cec_timing(track, n_pairs=200, seed=0):
    random.seed(seed)
    files = {r["sample_id"]: r["file"] for r in csv.DictReader(open(f"{ROOT}/samples.csv")) if r["track"] == track}
    pairs = [(g["query_id"], g["candidate_id"], g["label"]) for g in csv.DictReader(open(f"{ROOT}/eval/{track}_test_hard_gallery.csv"))]
    times, agree = [], 0
    for q, c, lab in random.sample(pairs, n_pairs):
        t = time.perf_counter()
        out = subprocess.run(["yosys-abc", "-c", f"cec {ROOT}/{files[q]} {ROOT}/{files[c]}"], capture_output=True, text=True).stdout
        times.append(time.perf_counter() - t)
        agree += (("are equivalent" in out) == (lab == "1"))
    n_test = sum(1 for r in csv.DictReader(open(f"{ROOT}/samples.csv")) if r["track"] == track and r["split"] == "test")
    return dict(track=track, pairs=n_pairs, cec_ms_per_pair=round(1000 * statistics.mean(times), 1),
                cec_correct=f"{agree}/{n_pairs}", full_search_per_query_s=round(statistics.mean(times) * (n_test - 1), 1))

import shutil
if shutil.which("yosys-abc"):
    cec = pd.DataFrame([cec_timing("synthetic"), cec_timing("real")])
    display(cec)
else:
    print("ABC not installed - run: !apt-get -qq install -y yosys")

,track,pairs,cec_ms_per_pair,cec_correct,full_search_per_query_s
0,synthetic,200,52.8,200/200,54.9
1,real,200,143.1,200/200,1271.7


## all results
saved to `shireen_baseline_results.csv`

In [12]:
results = pd.concat([syn, real], ignore_index=True)
results.to_csv("shireen_baseline_results.csv", index=False)
results

,track,split,condition,method,queries,hit1,mrr,r5,r10,ap,vectors_ms_per_circuit,rank_ms_per_query
0,synthetic,test,hard,histogram,201,0.423,0.588,0.234,0.393,0.360,0.001,1.718
1,synthetic,test,full,histogram,202,0.228,0.356,0.105,0.154,0.114,0.001,1.559
2,synthetic,test,hard,WL (h=2),201,0.821,0.879,0.403,0.526,0.533,0.018,1.939
3,synthetic,test,full,WL (h=2),202,0.713,0.753,0.269,0.313,0.296,0.018,2.311
4,synthetic,test,hard,WL + input positions (h=1),201,0.856,0.905,0.445,0.572,0.573,0.008,1.887
5,synthetic,test,full,WL + input positions (h=1),202,0.767,0.812,0.319,0.362,0.352,0.008,1.615
6,real,test,hard,histogram,1598,0.605,0.743,0.387,0.697,0.604,0.004,0.293
7,real,test,full,histogram,1652,0.368,0.456,0.144,0.187,0.159,0.004,2.133
8,real,test,hard,WL (h=2),1598,0.766,0.861,0.470,0.771,0.690,0.059,0.626
9,real,test,full,WL (h=2),1652,0.490,0.586,0.208,0.282,0.245,0.059,2.475
